# Sample Superstore — Exploratory Data Analysis and Visualization
**Course:** EDAV (22ADC32N) · Course-End Project  
**Workflow:** Data Loading → Data Cleaning → Analysis → Visualization → Conclusion

## Objective
Analyze the Sample Superstore dataset using NumPy, Pandas, Matplotlib, and Seaborn to explore sales, profit, product performance, regional differences, discounts, and monthly trends.

**How to use:** Run the cells from top to bottom in Google Colab. When prompted, upload `Sample_ Superstore.csv`.

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

## 2. Data Inspection

In [ ]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 3. Data Cleaning

In [ ]:
df = df.copy()

# Remove duplicate records
df.drop_duplicates(inplace=True)

# Convert date columns to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

# Convert numerical columns safely
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows missing essential analytical values
df.dropna(subset=["Order Date", "Sales", "Profit"], inplace=True)

# Create time-based features
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.strftime("%b")
df["Quarter"] = df["Order Date"].dt.quarter

# Calculate shipping duration; negative values are retained for inspection
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Cleaning completed.")
print("Cleaned dataset shape:", df.shape)
print("\nRemaining missing values:")
display(df.isnull().sum())
display(df.head())

## 4. NumPy Fundamentals and Summary Statistics

In [ ]:
sales = df["Sales"].to_numpy()
profit = df["Profit"].to_numpy()
quantity = df["Quantity"].to_numpy()

print("SALES ANALYSIS USING NUMPY")
print("Total Sales:", np.sum(sales))
print("Average Sales:", np.mean(sales))
print("Median Sales:", np.median(sales))
print("Minimum Sales:", np.min(sales))
print("Maximum Sales:", np.max(sales))
print("Sales Standard Deviation:", np.std(sales))

print("\nPROFIT ANALYSIS")
print("Total Profit:", np.sum(profit))
print("Average Profit:", np.mean(profit))
print("Median Profit:", np.median(profit))
print("Profitable transactions:", np.sum(profit > 0))
print("Loss-making transactions:", np.sum(profit < 0))

print("\nQUANTITY ANALYSIS")
print("Total Quantity:", np.sum(quantity))
print("Average Quantity:", np.mean(quantity))

## 5. Sales and Profit Analysis Using Pandas

In [ ]:
print("Overall Business Performance")

summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Profit",
        "Total Quantity Sold",
        "Average Sales per Record",
        "Average Profit per Record",
        "Profit Margin (%)"
    ],
    "Value": [
        df["Sales"].sum(),
        df["Profit"].sum(),
        df["Quantity"].sum(),
        df["Sales"].mean(),
        df["Profit"].mean(),
        (df["Profit"].sum() / df["Sales"].sum()) * 100
    ]
})
display(summary.round(2))

print("\nSales and Profit by Category")
category_analysis = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Average_Discount=("Discount", "mean")
).sort_values("Total_Sales", ascending=False)
display(category_analysis.round(2))

print("\nRegional Performance")
region_analysis = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).sort_values("Total_Sales", ascending=False)
display(region_analysis.round(2))

print("\nSegment Performance")
display(
    df.groupby("Segment")[["Sales", "Profit", "Quantity"]]
      .sum()
      .sort_values("Sales", ascending=False)
      .round(2)
)

## 6. Visualization — Sales and Profit by Category

In [ ]:
category_sales = df.groupby("Category")["Sales"].sum()
category_profit = df.groupby("Category")["Profit"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

category_sales.plot(kind="bar", ax=axes[0], title="Total Sales by Category")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Sales")
axes[0].tick_params(axis="x", rotation=30)

category_profit.plot(kind="bar", ax=axes[1], title="Total Profit by Category")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Profit")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 7. Visualization — Monthly Sales Trend

In [ ]:
monthly_sales = (
    df.groupby(df["Order Date"].dt.to_period("M"))["Sales"]
      .sum()
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales.index.astype(str), monthly_sales.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Visualization — Regional Performance

In [ ]:
regional_data = df.groupby("Region")[["Sales", "Profit"]].sum()

regional_data.plot(
    kind="bar", figsize=(10, 5), title="Regional Sales vs Profit"
)
plt.xlabel("Region")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Visualization — Correlation Heatmap

In [ ]:
numeric_data = df[
    ["Sales", "Quantity", "Discount", "Profit", "Shipping Days"]
].dropna()

plt.figure(figsize=(9, 6))
sns.heatmap(
    numeric_data.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Between Numerical Variables")
plt.tight_layout()
plt.show()

## 10. Top and Bottom Performing Products

In [ ]:
product_analysis = df.groupby("Product Name").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
)

print("Top 10 Products by Sales")
display(product_analysis.sort_values("Total_Sales", ascending=False).head(10).round(2))

print("\nTop 10 Products by Profit")
display(product_analysis.sort_values("Total_Profit", ascending=False).head(10).round(2))

print("\nBottom 10 Products by Profit")
display(product_analysis.sort_values("Total_Profit", ascending=True).head(10).round(2))

## 11. Discount and Profitability Analysis

In [ ]:
discount_analysis = df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean"),
    Transaction_Count=("Profit", "count")
).reset_index()
display(discount_analysis.round(2))

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x="Discount", y="Profit", hue="Category", alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Relationship Between Discount and Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

## 12. Automatically Generated Findings and Conclusion

In [ ]:
best_category = category_analysis["Total_Sales"].idxmax()
best_profit_category = category_analysis["Total_Profit"].idxmax()
best_region = region_analysis["Total_Sales"].idxmax()
best_profit_region = region_analysis["Total_Profit"].idxmax()
worst_product = product_analysis["Total_Profit"].idxmin()
best_product = product_analysis["Total_Profit"].idxmax()

loss_records = (df["Profit"] < 0).sum()
profit_margin = df["Profit"].sum() / df["Sales"].sum() * 100

print("=" * 55)
print("FINAL BUSINESS INSIGHTS")
print("=" * 55)
print(f"1. Total sales: ${df['Sales'].sum():,.2f}")
print(f"2. Total profit: ${df['Profit'].sum():,.2f}")
print(f"3. Overall profit margin: {profit_margin:.2f}%")
print(f"4. Highest-sales category: {best_category}")
print(f"5. Highest-profit category: {best_profit_category}")
print(f"6. Highest-sales region: {best_region}")
print(f"7. Highest-profit region: {best_profit_region}")
print(f"8. Most profitable product: {best_product}")
print(f"9. Lowest-profit product: {worst_product}")
print(f"10. Loss-making records: {loss_records}")

print("\nCONCLUSION")
print(
    "The analysis examines sales, profitability, product performance, "
    "regional differences, monthly trends, discounts, and shipping. "
    "The findings can help identify high-performing categories, "
    "unprofitable products, and opportunities to improve business "
    "performance. Recommendations should be based on the calculated "
    "results rather than sales alone."
)

## 13. Save the Cleaned Dataset (Optional)
Run this cell if you want to download the cleaned dataset.

In [ ]:
output_file = "Sample_Superstore_Cleaned.csv"
df.to_csv(output_file, index=False)
files.download(output_file)